<a href="https://colab.research.google.com/github/h3it-dias/FastCamp_Agent_AI_CEIA/blob/main/Agents_ADK/Agents_ADK_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema multi-agent para automação de relatórios
O sistema vem com uma arquitetura multi-agent, com um Agente orquestrador e dois Sub-Agents especializados. O objetivo em quetão é o Agente orquestrador coordenar o agente de obteção de informações e o agente de resumo de informações para produzir um relatório acerca de determinado assunto.

Instalação das bibliotecas

In [ ]:
!pip install google-adk -q
!pip install litellm -q
!pip install groq

Importação das bibliotecas necessárias para o desenvolvimento

In [ ]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from typing import Optional
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from groq import Groq

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

Configuração da API Key do Groq que iremos usar no futuro

In [ ]:
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

print("API Keys Set:")

print(f"GROQ API Key set: {'Yes' if os.environ.get('GROQ_API_KEY') != 'YOUR_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")

Criação das ferramentas necessárias para os agentes usarem. Nesse caso essas ferramentas possuem, cada uma, um modelo de linguagem para usarem na sua ação. Como dito antes, as ferramentas buscam obter informações (obter_informacoes) e resumir informações (resumir_informacoes).

In [ ]:
def obter_informacoes(area_busca: str) -> str:
  """
  Ferramenta de IA secundária para obter as informações a acerca da {area_busca}
  """
  client = Groq()

  prompt_interno = f"""Leia a area de busca a seguir e RETORNE um texto com as informações principais encontradas. Area de busca: {area_busca}"""

  try:
    chamada_modelo = client.chat.completions.create(
        messages=[
              {
                  "role": "system",
                  "content": "Você é um especialista em pesquisar assuntos de diversas areas"
              },
              {
                  "role": "user",
                  "content": prompt_interno
              }
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.1,
    )

    resposta_modelo = chamada_modelo.choices[0].message.content
    return resposta_modelo

  except Exception as e:
    print(f"Erro na ferramenta: {e}")
    return {"status": "erro", "mensagem": str(e)}

def resumir_informacoes(informacoes: str) -> str:
  """
  Ferramenta de IA secundária para resumir as informações obtidas
  """
  client = Groq()

  prompt_interno = f"""Leia as informações a seguir e RETORNE um texto resumido do assunto. Informações: {informacoes}"""

  try:
    chamada_modelo = client.chat.completions.create(
        messages=[
              {
                  "role": "system",
                  "content": "Você é um especialista em resumir textos"
              },
              {
                  "role": "user",
                  "content": prompt_interno
              }
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.1,
    )

    resposta_modelo = chamada_modelo.choices[0].message.content
    return resposta_modelo

  except Exception as e:
    print(f"Erro na ferramenta: {e}")
    return {"status": "erro", "mensagem": str(e)}



Criação dos agentes que iremos usar ao longo dos sistema. A arquitetura está formada da seguinte forma: Um agente orquestrador(root_agent_relatorio) coordenando dois sub-agentes especializados (agent_pesquisador e agent_resumidor). Também é nessa etapa que inicializamos uma memória para os agentes, por meio de servico_memoria = InMemorySessionService(), e criamos nosso runner para conduzir do o fluxo.

In [ ]:
try:
    agent_pesquisador = Agent(
        model=LiteLlm(model="groq/llama-3.3-70b-versatile"),
        name="agent_pesquisador",
        instruction="Você é um agente de pequisa. Sua ÚNICA tarefa é obter as informações de um determinado assunto usando a ferramenta 'obter_informacoes'. Não tente realizar nenhum outra ação",
        description="Opera com pesquisa para encontrar informações usando a ferramenta 'obter_informacoes'",
        tools=[obter_informacoes],
    )
    print(f"Sub-Agent '{agent_pesquisador.name}' criado.")
except Exception as e:
    print(f"Não foi possível criar o agente pesquisador. Error: {e}")

try:
    agent_resumidor = Agent(
        model=LiteLlm(model="groq/llama-3.3-70b-versatile"),
        name="agent_resumidor",
        instruction="Você é um agente de síntese de textos. Sua ÚNICA tarefa é pegar as informações brutas recebidas e usar a ferramenta 'resumir_informacoes' para criar um texto curto e claro. Não adicione informações externas ou realize outra ação.",
        description="Opera exclusivamente para resumir blocos de texto usando a ferramenta 'resumir_informacoes'",
        tools=[resumir_informacoes],
    )
    print(f"Sub-Agent '{agent_resumidor.name}' criado.")
except Exception as e:
    print(f"Não foi possível criar o agente pesquisador. Error: {e}")

try:
    root_agent_model = LiteLlm(model="groq/llama-3.1-8b-instant", max_tokens=500)

    root_agent_relatorio = Agent(
        name="root_agent_V1_relatorio",
        model=root_agent_model,
        description="Agente principal: Recebe solicitações do usuário, gerencia o fluxo de trabalho e delega as tarefas para os agentes especialistas ('agent_pesquisador' e 'agent_resumidor').",
        instruction="""
                  Você é o agente orquestrador.

                  Fluxo obrigatório:
                  1. Envie a pergunta do usuário para agent_pesquisador.
                  2. Quando receber o resultado, envie para agent_resumidor.
                  3. Retorne o resumo final ao usuário.

                  REGRAS:
                  - Nunca chame root_agent_V1_relatorio
                  - Nunca execute tarefas diretamente
                  - Apenas delegue para agent_pesquisador ou agent_resumidor
        """,
        sub_agents=[agent_pesquisador, agent_resumidor],
    )
    print(f"Root Agent '{root_agent_relatorio.name}' criado.")

except Exception as e:
    print(f"Não foi possível criar o agente orquestrador. Error: {e}")


servico_memoria = InMemorySessionService()

try:
    runner_root = Runner(
        agent=root_agent_relatorio,
        app_name="SistemaPesquisaResumo",
        session_service=servico_memoria
    )
    print(f"Runner criado e configurado para o agente.")
except Exception as e:
    print(f"Não foi possível criar o runner. Error: {e}")

Nessa etapa temos a criação de uma função chamar_agent, que é responsável por enviar a pergunta para o agente no formato ideal e retornar a resposta final.

In [ ]:
async def chamar_agent(query: str, runner, user_id, session_id):
    """Envia a pergunta para o agente e retorna a resposta final."""

    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    final_response_text = "O agente não produziu resposta."

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):

        if event.content and event.content.parts:
            texto = event.content.parts[0].text
            if texto:
                final_response_text = texto

    return final_response_text

Por fim, para executar todo o sistema, temos essa parte do código que fica responsável por iniciar o chat (iniciar_chat) passando a informação do usuário e chamando a função chamar_agent.

In [ ]:
import asyncio

async def iniciar_chat():
    print("SISTEMA DE PESQUISA MULTI-AGENTE INICIADO")

    USER_ID = "usuario_teste_123"
    SESSION_ID = "sessao_001"
    await servico_memoria.create_session(
        app_name="SistemaPesquisaResumo",
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    entrada_usuario = input("Você: ")

    print("Orquestrador delegando tarefas")

    try:
        resposta = await chamar_agent(
                entrada_usuario,
                runner_root,
                USER_ID,
                SESSION_ID
        )

        print(f"\nOrquestrador: {resposta}\n")

    except Exception as e:
        print(f"\nErro durante a execução: {e}\n")


await iniciar_chat()